In [ ]:

%load_ext ElasticNotebook

# CAB AGGLOMERATOR FAST CARS CASE STUDY<br>

## INTRODUCTION<br>

Suppose you are working as a data analyst in a company called Fast Cars. Fast Cars is a cab agglomerator like Uber and Ola i.e. it connects passengers to cabs in cities for travel through an app.

They are going to launch their product in New York City. But before they want to understand the new york taxi market.

As a data analyst you are provided with a Yellow Taxi dataset which contains information about taxis that people took in New York city from streets. 

You are asked to analyse this data to provide insights about the taxi market of new york.

## DATA IMPORT AND BASIC OBSERVATIONS

In [ ]:
%%RecordEvent
# %load_ext cudf.pandas
# import important libraries - matplotlib, seaborn and pandas
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_0.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 0 ###

file_loc = '/home/dias-benchmarks/new_notebooks/nyc-taxi/yellow_tripdata.csv'

# read file
trip_data = pd.read_csv(file_loc)
trip_data = trip_data[:100000]
trip_data.shape, trip_data.head()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_1.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 1 ###

# print data tail
trip_data.tail()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_2.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 2 ###

# print data info
trip_data.info()

By looking at data_description file we find that following columns would be useful for our analysis:<br>
* tpep_pickup_datetime - The date and time when the meter was engaged. 
* tpep_dropoff_datetime - The date and time when the meter was disengaged.
* Passenger_count - The number of passengers in the vehicle. 
* Trip_distance - The elapsed trip distance in miles reported by the taximeter.
* PULocationID - TLC Taxi Zone in which the taximeter was engaged
* DOLocationID - TLC Taxi Zone in which the taximeter was disengaged
* Payment_type - A numeric code signifying how the passenger paid for the trip. 
* Fare_amount - The time-and-distance fare calculated by the meter.
* Extra - Miscellaneous extras and surcharges. Currently, this only includes the $0.50 and $1 rush hour and overnight charges.
* MTA_tax - \\$ 0.50 MTA tax that is automatically triggered based on the metered rate in use.
* Improvement_surcharge - \\$0.30 improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015.
* congestion_surcharge - fees on congestion
* Tip_amount - This field is automatically populated for credit card tips. Cash tips are not included.
* Tolls_amount - Total amount of all tolls paid in trip.
* Total_amount -  The total amount charged to passengers. Does not include cash tips.

And we will drop the following columns:<br>
* VendorID
* RateCodeID 
* Store_and_fwd_flag

## DATA MANIPULATION BEFORE EDA

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_3.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 3 ###

# remove following columns - 'VendorID','RatecodeID','store_and_fwd_flag'
trip_data.drop(['VendorID','RatecodeID','store_and_fwd_flag'],axis=1,inplace=True)
# print data head
trip_data.head()

We will now deal with time related columns, we have two time related columns
* tpep_pickup_datetime 
* tpep_dropoff_datetime

We will first convert these column to datatime data type of pandas.

we will create three different features from these  
* hour - pickup hour and dropoff hour
* day name - this is basically the day of the week when trip took place - we will only take day name from pickup date.
( as day name for drop date is supposed to be same as pickup date)
* duration of trip

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_4.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 4 ###

# convert 'tpep_pickup_datetime' and 'tpep_dropoff_datetime' to datetime format
trip_data['tpep_pickup_datetime'] = pd.to_datetime(trip_data['tpep_pickup_datetime'],errors='coerce' )
trip_data['tpep_dropoff_datetime'] = pd.to_datetime(trip_data['tpep_dropoff_datetime'], errors='coerce' )
# print data info
print(trip_data.info())
# print data head
trip_data.head()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_5.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 5 ###

# create 'duration' column using pd.Timedelta(minutes=1)
trip_data['duration'] = (trip_data['tpep_dropoff_datetime'] - trip_data['tpep_pickup_datetime'])/ pd.Timedelta(minutes=1)
# create 'trip_pickup_hour' column using 'tpep_pickup_datetime' column
trip_data['trip_pickup_hour'] = trip_data['tpep_pickup_datetime'].dt.hour
# create 'trip_dropoff_hour' column using 'tpep_dropoff_datetime' column
trip_data['trip_dropoff_hour'] = trip_data['tpep_dropoff_datetime'].dt.hour
# create 'trip_day' column using 'tpep_pickup_datetime' column - use day_name()
trip_data['trip_day'] = trip_data['tpep_pickup_datetime'].dt.day_name()
# print data info
print(trip_data.info())
# print data head
trip_data.head()

Let's also see the number of missing values for each column

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_6.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 6 ###

# print missing values for each column - use .isnull().sum
trip_data.isnull().sum(axis=0).reset_index()

From the above table we can observe that we have no missing values.

For payment_type we have the following mapping for categories:<br>
1= Credit card
2= Cash
3= No charge
4= Dispute
5= Unknown
6= Voided trip

let's just check if we have only these categories available in payment_type or not

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_7.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 7 ###

# value_counts for 'payment_type' column
trip_data['payment_type'].value_counts()

Now we will replace these number in payment category with actual category names.

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_8.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 8 ###

# function for mapping numerical payment_type to actual payment
def map_payment_type(x):
    if x==1:
        return 'Credit_card'
    elif x==2:
        return 'Cash'
    elif x==3:
        return 'No_charge'
    elif x==4:
        return 'Dispute'
    elif x==5:
        return 'Unknown'
    else:
        return 'Voided_trip'

# use .apply and lambda on payment_type column to change 'payment_type' column
trip_data['payment_type'] = trip_data.payment_type.apply(lambda x:map_payment_type(x))
# print data head
trip_data.head()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_9.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 9 ###

# print data info to show that payment_type data type has changed
trip_data.info()

Now our Total_amount is basically<br>
Total_amount = fare_amount + tolls_amount + tip_amount + (extra + mta_tax + improvement_surcharge)

of the above components of total_amount we will specifically focus on 'fare_amount','tip_amount', 'tolls_amount' and 'total taxes'.

We are combining the extra, mta_tax and improvement_surcharge under one category called total_taxes as these are determined by local laws and taxes and is not dependent upon distance travelled or time taken for trip.

Here total taxes would be the sum of three columns 'extra','mta_tax', 'improvement_surcharge'. So we will make a new column for total_taxes.

We will also drop these three columns 'extra','mta_tax','improvement_surcharge'.


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_10.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 10 ###

# create 'total_taxes' column from summing 'extra','mta_tax', 'improvement_surcharge'
trip_data['total_taxes'] = trip_data['extra']+trip_data['mta_tax']+trip_data['improvement_surcharge']
# drop 'extra','mta_tax','improvement_surcharge' columns
trip_data.drop(['extra','mta_tax','improvement_surcharge'],axis=1,inplace=True)
# print data head
trip_data.head()

## ASSUMPTIONS/ANALYSIS THAT MIGHT BE USEFUL FOR OUR COMPANY<br>

**IMPORTANT CHARACTERISTICS OF A TRIP**
* fare_amount, trip_distance, duration and passenger count distribution can tell us about the important characteristics about the trip.


**PRICING EXPLORATION**
* payment_type can tell us which kind of payment mode the customer usually favours.


* Another issue that taxi companies face is pricing the trip appropriately. So for exploring the pricing of trip, we can also look into the relationship between pricing related variables and hour/day of trip and pricing related variables and location.

**TIME/LOCATION EXPLORATION** 
* To maximize the earnings we should be focussing on trips which are on busy locations and busy times. 


**DURATION OF TRIP EXPLORATION**
* A typical taxi company faces a common problem of efficiently assigning the cabs to passengers so that the service is smooth and hassle free. One of main issue is determining the duration of the current trip. So, We should look into relationship between duration and location, duration and hour of trip.




## DATA ANALYSIS<br>

**UNIVARIATE ANALYSIS**<br>
The first step in doing any kind of EDA is identifying the distribution of important variables in EDA. This helps us in finding important insights about the data.<br>
We should look into the distribution of these specific columns:<br>
Price Based Columns
* fare_amount
* tip_amount
* total_taxes
* tolls_amount
* payment_type
* total_amount

Time Based Columns
* duration
* trip_pickup_hour
* trip_dropoff_hour
* trip_day

Distance/Location Based Columns
* trip_distance
* PULocationID
* DOLocationID

Other columns
* passenger_count

Before we explore the distribution of each column we must identify column category because distribution analysis depends upon variable category:<br>
* Continuous - column which are measurable and uncountable in nature - we use histograms and box plot
* Categorical - column which have categories as it data - we use bar charts

Following columns are continuous in nature:<br>
* fare_amount
* tip_amount
* total_taxes
* total_amount
* duration
* trip_distance
* tolls_amount

Following columns are categorical in nature:<br>
* payment_type
* trip_pickup_hour - it has 24 categories
* trip_dropoff_hour - it has 24 categories
* trip_day - it has 7 categories
* PULocationID
* DOLocationID
* Passenger_count

We will look into the distrbution of passenger_count at the last.

**CONTINUOUS VARIABLE DISTRIBUTION**

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_11.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 11 ###

# continuous_columns list
continuous_columns = ['fare_amount','tip_amount','total_taxes','total_amount','duration','trip_distance','tolls_amount']

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_12.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 12 ###

# use .describe() for showing the statistics for continuous columns
trip_data[continuous_columns].describe()

In [15]:
# # import gridspec function from matplotlib
# from matplotlib import gridspec

In [16]:
# # for loop for continuous_columns variable
# for feature in continuous_columns:
#     # create a figure object using plt.figure
#     fig = plt.figure(figsize=(14,7))
#     # use gridspec.GridSpect with arguments nrows, ncols and figure to create areas for 2 plots in figure object
#     gs = gridspec.GridSpec(nrows=1, ncols=2, figure=fig)
#     # ax1 is first axes object created using fig.add_subplot(gs[0,0]) - controls the first area of figure object
#     ax1 = fig.add_subplot(gs[0, 0])
#     # histogram plot in first area using sns.distplot - attributes are kde and ax
#     sns.distplot(trip_data[feature],kde=True,ax=ax1)
#     # using ax1.set_title to create title for histograms
#     ax1.set_title('histogram of column values in '+feature)
#     # ax2 is second axes object created using fig.add_subplot(gs[0,1]) - controls the second area of figure object
#     ax2 = fig.add_subplot(gs[0,1])
#     # box plot  in second area using sns.boxplot - attributes are ax
#     sns.boxplot(trip_data[feature],ax=ax2)
#     # using ax2.set_title for box plot
#     ax2.set_title('box plot of column values in '+feature)
#     # seaborn style setting
#     sns.set()
#     # matplotlib command for displaying plots
#     plt.show()

Negtive values for columns does not make sense<br>
fare_amount<br>
tip_amount<br>
total_taxes<br>
tolls_amount<br>
total_amount<br>
duration<br>

Let's just observe how the negative values in each of these columns look like

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_13.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 13 ###

# using .loc to show negative values in fare_amount
trip_data.loc[trip_data['fare_amount']<0]

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_14.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 14 ###

# using .loc to show negative values in tip_amount
trip_data.loc[trip_data['tip_amount']<0]

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_15.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 15 ###

# using .loc to show negative values in tolls_amount
trip_data.loc[trip_data['tolls_amount']<0]

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_16.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 16 ###

# using .loc to show negative values in total_taxes
trip_data.loc[trip_data['total_taxes']<0]

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_17.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 17 ###

# using .loc to show negative values in total_amount
trip_data.loc[trip_data['total_amount']<0]

From the above table displays it is clear whenever fare_amount is negative, we have negative values in 'tip_amount','total_taxes' and 'total_amount'. Negative values for these cases does not make sense for doing our analysis. The reason for these negative values can be explored later on if we want to understand the data more better. For now we will remove these rows. 

Also, number of negative rows are 4260 which is 0.04% of total 8759874 observations. So even if we remove them it does not hamper the quantity of data that we have.

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_18.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 18 ###

# data shape before filtering negative fare_amount rows
print(trip_data.shape)
# using .loc to filter only those rows where fare_amount is positive 
trip_data = trip_data.loc[trip_data['fare_amount']>=0]
trip_data.shape, trip_data.head()

We will look into the negative values for duration

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_19.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 19 ###

# using .loc to show negative values in duration
trip_data.loc[trip_data['duration']<0]

Since there are only two rows with negative duration, we will remove them so as to do our analysis in a better way

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_20.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 20 ###

# using .loc to filter only those rows where duration is positive 
trip_data = trip_data.loc[trip_data['duration']>=0]
trip_data.shape

Now we will again look at the distribution plots for these variables

In [25]:
# # plot box plot and histograms for continuous_columns variables
# for feature in continuous_columns:
#     fig = plt.figure(figsize=(14,7))
#     gs = gridspec.GridSpec(nrows=1, ncols=2, figure=fig)
#     ax1 = fig.add_subplot(gs[0, 0])
#     sns.distplot(trip_data[feature],kde=True,ax=ax1)
#     ax1.set_title('histogram of column values in '+feature)
#     ax2 = fig.add_subplot(gs[0,1])
#     sns.boxplot(trip_data[feature],ax=ax2)
#     ax2.set_title('box plot of column values in '+feature)
#     sns.set()
#     plt.show()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_21.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 21 ###

# use .describe() again to show the statistics for these continuous variables
trip_data[continuous_columns].describe()

Looking from the above histograms we can decipher following information for each column <br>
* fare_amount  - most of the fare amount is within 10 dollar value as is shown by the median value. Though there are some significant outliers, the maximum of which is 8000 dollars.


* tip_amount - most of the tip amount is within 1-2 dollar as is shown by the median value. Though again here too we have outliers, the maximum of which is around 450 dollars. 


* tolls_amount - most of the tolls_amount value is 0 so it seems most of the trips do not have to pay for tolls.


* total_taxes - most of the total_taxes values is within 0.8 dollars as is shown by the median value. Though we have outliers in this case but it is not as signiificant as the case for tip and fare.


* total_amount - most of the total_amount values is within 11 dollars as is shown by the median value. Again the outliers in this case seems mostly because of outliers in fare_amount.


* duration - most of the values in duration is within 10 minutes range as is shown by the median value. We do have some outliers which are beyond the range of 1000 minutes.


* trip_distance - most of the trip_distance is within 1.55 miles value as is shown by the median. The outlier in this case is quite less.

**CATEGORICAL VARIABLE DISTRIBUTION**<br>
Let's move on to analyse the distribution of categorical variables

for analysing distribution of categorical variables we use bar plots showing the count% of each category.

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_22.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 22 ###

# list of categorical_variables
categorical_variables = ['payment_type','trip_pickup_hour','trip_dropoff_hour','trip_day','PULocationID','DOLocationID']

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_23.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 23 ###

# start exploration with payment_type using .value_counts()
trip_data['payment_type'].value_counts()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_24.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 24 ###

# but this is a series for ease of plotting we need to use dataframe using .reset_index() on value_counts()
payment_type_category_count = trip_data['payment_type'].value_counts().reset_index()
payment_type_category_count.columns = ["payment_type", "count"]
# print the above dataframe
payment_type_category_count

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_25.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 25 ###

# we are shown the count under each category but it is better to have count% for comparison - create count_percent col
payment_type_category_count['count_percent'] = (payment_type_category_count['count']/trip_data.shape[0])*100
# print the data frame
payment_type_category_count

In [31]:
# # now let's plot it as bar chart
# # first step - create fig, ax object using plt.subplots
# fig,ax = plt.subplots(figsize=(7,7))
# # second step - use sns.barplot(x, y , data, ax) for plotting bar plot
# sns.barplot(x = 'index', y = 'count_percent', data=payment_type_category_count,ax=ax)
# # third step - use ax object to change plot properties - here we set a title with ax.set_title()
# ax.set_title('box plot for payment_type column')
# # third step - seaborn style setting
# sns.set()
# # fourth step - use plt.show() for showing the plots
# plt.show()

From above we can understand that most of the payments are done through cash and credit cards. The proportion of credit card payments is around 70%.

Now we look into time based categorical variables.<br>
* 'trip_pickup_hour'
* 'trip_dropoff_hour'
* 'trip_day'


In [32]:
# # now let's plot all the time based categorical variables in this way using a for loop
# for feature in ['trip_pickup_hour','trip_dropoff_hour','trip_day']:
#     # Create a dataframe for the feature using value_counts().reset_index()
#     feature_value_counts = trip_data[feature].value_counts().reset_index()
#     # create count_percent column
#     feature_value_counts['count_percent'] = (feature_value_counts[feature]/trip_data.shape[0])*100
#     # print the number of categories in the feature
#     print('Number of categories in feature '+ feature + ' is ' + str(feature_value_counts.shape[0]))
#     # Create fig,ax object using plt.subplots
#     if feature_value_counts.shape[0]<10:
#         fig,ax = plt.subplots(figsize=(7,7))
#     else:
#         fig,ax = plt.subplots(figsize=(20,7))
#     # plot barplot x='index' and y='count_percent' using sns.barplot
#     sns.barplot(x='index',y='count_percent',data=feature_value_counts,ax=ax)
#     # set_title
#     ax.set_title('Bar plot for '+ feature)
#     # set_xlabel
#     ax.set_xlabel(feature)
#     sns.set()
#     plt.show()

Based on above plots we can observe following things
* Trip Hour 
    * the dropoff and pick up hour distribution looks almost same, it is because the trip duration in most of the cases is less than an hour with the median duration value as 10 min. 
    * Peak hour for the pick up and drop off is around evening from 5 to 8. The busiest time is 6PM.
    * there is less traffic during night times and only after 8AM in morning does the pickup and drop off starts picking up pace.
* Trip day
    * Sunday has the lowest taxi uses.
    * Weekdays except thursday have heavy taxi uses.
    * Among weekends Staturday has taxi uses on the same level as weekdays.


Moving on we will explore the distribution of location based features:<br>
* 'PULocationID'
* 'DOLocationID'

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_26.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 26 ###

# let's see the number of categories available in both pickup and dropoff location - PULocationID and DOLocationID
trip_data['PULocationID'].value_counts().shape, trip_data['DOLocationID'].value_counts().shape

So we have around 260 categories for location. To plot it on bar plots we need to increase the figure size.


In [34]:
# for feature in ['PULocationID','DOLocationID']:
#     # Create a dataframe for the feature using value_counts().reset_index()
#     feature_value_counts = trip_data[feature].value_counts().reset_index()
#     # create count_percent column
#     feature_value_counts['count_percent'] = (feature_value_counts[feature]/trip_data.shape[0])*100
#     # print the number of categories in the feature
#     print('Number of categories in feature '+ feature + ' is ' + str(feature_value_counts.shape[0]))
#     # Create fig,ax object using plt.subplots
#     fig,ax = plt.subplots(figsize=(25,7))
#     # plot barplot x='index' and y='count_percent' using sns.barplot
#     sns.barplot(x='index',y='count_percent',data=feature_value_counts,ax=ax)
#     # set_title
#     ax.set_title('Bar plot for '+ feature)
#     # set_xlabel
#     ax.set_xlabel(feature)
#     sns.set()
#     plt.show()

The above plots looks quite messy but one insight that we can indetify from above plot that most of pickup and dropoff points do not have more 0.5% traffic (0.5 percent of 8755612 total trips is 43778).

So in our next plot we will filter out these pickup and dropoff points to look into the graph more clearly.

In [35]:
# for feature in ['PULocationID','DOLocationID']:
#     feature_value_counts = trip_data[feature].value_counts().reset_index()
#     feature_value_counts['count_percent'] = (feature_value_counts[feature]/trip_data.shape[0])*100
#     # filter only those location which has more than 0.5 % of traffic
#     feature_value_counts = feature_value_counts.loc[feature_value_counts['count_percent']>=0.5]
#     print('Number of categories in feature '+ feature + ' above 0.5 % count is ' + str(feature_value_counts.shape[0]))
#     fig,ax = plt.subplots(figsize=(25,7))
#     sns.barplot(x='index',y='count_percent',data=feature_value_counts,ax=ax)
#     ax.set_title('Bar plot for '+ feature)
#     ax.set_xlabel(feature)
#     sns.set()
#     plt.show()

From the above plots we can glance following insights<br>
* The busiest location in terms of pickup are 161, 236 and 237
* The busiest location for dropoff too are 161, 236 and 237.

We can also look for routes which are busiest. 

For exploring busy routes we need to create a new route column which is a combination of pickup and dropoff point.

So route = 'PULocationID'-'DULocationID'

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_27.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 27 ###

# create routes column using PULocationID and DOLocationID with lambda function
trip_data['routes'] = trip_data.apply(lambda x: str(x['PULocationID'])+'-'+str(x['DOLocationID']),axis=1)

Since the above code takes a lot of time to execute we will import already created routes data based on the above code and then merge it with the trip_data dataframe.

In [37]:
# # save routes data to csv to load it later for analysis
# trip_data['routes'].to_csv('/home/dias-benchmarks/new_notebooks/nyc-taxi/routes_yellow_tripdata_2018-01.csv',index=False)

In [38]:
# # loading routes_data
# file_loc_routes_data = 'routes_yellow_tripdata_2018-01.csv'
# routes_data = pd.read_csv(file_loc_routes_data)
# # assigning new column 'routes' in trip_data to routes_data
# trip_data['routes'] = routes_data

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_28.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 28 ###

trip_data.head()

Now let's explore routes through the same bar plot code that we used for Location ID's. But in this case we will only look for routes with more than 0.25% counts (21889 trips).

In [40]:
# # plot bar plot for routes which have trip count above 0.25%
# feature = 'routes'
# feature_value_counts = trip_data[feature].value_counts().reset_index()
# feature_value_counts['count_percent'] = (feature_value_counts[feature]/trip_data.shape[0])*100
# # choosing routes where the trip percent is above 0.25% of total trips
# feature_value_counts = feature_value_counts.loc[feature_value_counts['count_percent']>=0.25]
# print('Number of categories in feature '+ feature + ' above 0.25 % count is ' + str(feature_value_counts.shape[0]))
# fig,ax = plt.subplots(figsize=(25,7))
# sns.barplot(x='index',y='count_percent',data=feature_value_counts,ax=ax)
# ax.set_title('Bar plot for '+ feature)
# ax.set_xlabel(feature)
# sns.set()
# plt.show()

From the above plot we can observe that 5 busiest route are following:<br>
264-264<br>
237-236<br>
236-236<br>
236-237<br>
237-137<br>



Finally we will look into the distribution of passenger_count

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_29.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 29 ###

# look into value_counts of 'passenger_count'
trip_data['passenger_count'].value_counts()

Here we see that the mostly 1 or 2 passengers avail the cab. The instance of large group of people travelling together is rare.

This is the result of some of the most important insights after doing univariate analysis:<br>
* fare_amount  - most of the fare amount is within 10 dollar value as is shown by the median value. Though there are some significant outliers, the maximum of which is 8000 dollars.


* tip_amount - most of the tip amount is within 1-2 dollar as is shown by the median value. Though again here too we have outliers, the maximum of which is around 450 dollars. 


* duration - most of the values in duration is within 10 minutes range as is shown by the median value. We do have some outliers which are beyond the range of 1000 minutes.


* trip_distance - most of the trip_distance is within 1.55 miles value as is shown by the median. The outlier in this case is quite less.


* Credit card is the most preferred mode of payment followed by cash.


* Peak hour for the pick up and drop off is around evening from 5 to 8. The busiest time is 6PM.


* Weekdays except thursday have heavy taxi uses and among weekends Staturday has taxi uses on the same level as weekdays.


* The busiest location in terms of pickup and dropoff are 161, 236 and 237.


* Four of the busiest routes are - 264-264, 237-236, 236-236, 236-237


* Mostly 1 or 2 passenger avail the cab. Group rides are less common.

**BIVARIATE ANALYSIS**<br>
Remember that we made some analysis points regarding exploration of duration and pricing:<br>

For pricing we will be exploring it's relationship with:<br>
* hour/day of trip 
* pickup location of trip

For duration we will be exploring it's relationship with:<br>
* hour of day 
* pickup location of trip


**PRICING EXPLORATION**

We have following variables in the dataset that is associated with pricing:<br>
* fare_amount
* tip_amount
* total_taxes
* tolls_amount
* total_amount

In our anlaysis for now we will be focussing on:<br>
* fare_amount
* tip_amount
* total_taxes
* total_amount

we are leaving tolls_amount for now from our analysis as it contributes very little to the total_amount value because it's median value was 0 i.e. most of the trips are not paying tolls_amount.


*** PRICING VARIABLE EXPLORATION WITH HOUR/DAY OF TRIP ***<br>
All of our pricing variables are continuous and Hour/Day is categorical.

The way to explore relationship between a continuous variable and categorical variable is through a box plot. We create box plot for each category of categorical variable so as to see how the distribution changes for the continuous variables as the category values changes for categorical variable.

We will start with fare_amount exploration.

Let's do a box plot of fair_amount with hour/day of trip to see how the fare changes for different hours of the day and for different days of the week

In [42]:
# # fig,ax object using plt.subplots()
# fig,ax = plt.subplots(figsize=(25,7))
# # box plot using - sns.boxplot(x, y , data, ax)
# sns.boxplot(x = 'trip_pickup_hour',y='fare_amount',data=trip_data,ax=ax)
# # ax.set_title
# ax.set_title('box plot of fare_amount wrt hour of the day')
# # seaborn style setting
# sns.set()
# # matplotlib plt.show()
# plt.show()

In [43]:
# # fig,ax object using plt.subplots()
# fig,ax = plt.subplots(figsize=(25,7))
# # box plot using - sns.boxplot(x, y , data, ax)
# sns.boxplot(x = 'trip_dropoff_hour',y='fare_amount',data=trip_data,ax=ax)
# # ax.set_title
# ax.set_title('box plot of fare_amount wrt hour of the day')
# # seaborn style setting
# sns.set()
# # matplotlib plt.show()
# plt.show()

From the above plot we can observe that most of the outliers in fare happens during 15 or 3PM to 19 or 7PM based on pickup time.

Based on dropoff time, we have heavy outliers in the morning as well as evening.

For observing the distribution in a better way we would restrict the fare_amount to below 50 dollars. 

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_30.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 30 ###

# restricted_fare_amount_data dataframe formation by filtering fare_amount less than 50 dollars
restricted_fare_amount_data = trip_data.loc[trip_data['fare_amount']<=50]
restricted_fare_amount_data.shape

In [45]:
# fig,ax = plt.subplots(figsize=(25,7))
# sns.boxplot(x = 'trip_pickup_hour',y='fare_amount',data=restricted_fare_amount_data,ax=ax)
# ax.set_title('box plot of fare_amount wrt hour of the day')
# sns.set()
# plt.show()

In [46]:
# fig,ax = plt.subplots(figsize=(25,7))
# sns.boxplot(x = 'trip_dropoff_hour',y='fare_amount',data=restricted_fare_amount_data,ax=ax)
# ax.set_title('box plot of fare_amount wrt hour of the day')
# sns.set()
# plt.show()

We can see from the plots that trip pickup and dropoff hours do not have much affect on median fare_amount as median is almost same for all the hours.

let's us see if hour of day has any effect on other pricing related variables or not.

Starting with total_amount

In [47]:
# fig,ax = plt.subplots(figsize=(25,7))
# # sns.boxplot changes
# sns.boxplot(x = 'trip_pickup_hour',y='total_amount',data=trip_data,ax=ax)
# ax.set_title('box plot of total_amount wrt hour of the day')
# sns.set()
# plt.show()

In [48]:
# fig,ax = plt.subplots(figsize=(25,7))
# # sns.boxplot changes
# sns.boxplot(x = 'trip_dropoff_hour',y='total_amount',data=trip_data,ax=ax)
# ax.set_title('box plot of total_amount wrt hour of the day')
# sns.set()
# plt.show()

Again here since we are plotting full range of total_amount our graph is able to show heavy outliers prominently but not the distribution of general cases.

So we will again build a dataframe for total_amount with restricted values less than 50 dollars

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_31.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 31 ###

# restricted_total_amount_data for filtering total_amount data to less than 50 dollars
restricted_total_amount_data = trip_data.loc[trip_data['total_amount']<=50]
restricted_total_amount_data.shape

In [50]:
# fig,ax = plt.subplots(figsize=(25,7))
# sns.boxplot(x = 'trip_pickup_hour',y='total_amount',data=restricted_total_amount_data,ax=ax)
# ax.set_title('box plot of total_amount wrt hour of the day')
# sns.set()
# plt.show()

In [51]:
# fig,ax = plt.subplots(figsize=(25,7))
# sns.boxplot(x = 'trip_dropoff_hour',y='total_amount',data=restricted_total_amount_data,ax=ax)
# ax.set_title('box plot of total_amount wrt hour of the day')
# sns.set()
# plt.show()

Again we can see the median value does not changes much for each hour though there is variability in price across the hours indicated by different sizes of boxes for different hours.

We will explore tip_amount and total_taxes now. But for exploring them we will retrict the values for these variables to below 10 dollars because the median value for tip_amount was around 1-2 dollars while for total_taxes was around 0.8 dollars so if to see the general distribution clearly we are restricting it to a range of 5 times the median value.

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_32.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 32 ###

restricted_tip_amount_data = trip_data.loc[trip_data['tip_amount']<10]
restricted_total_taxes_data = trip_data.loc[trip_data['total_taxes']<10]
restricted_tip_amount_data.shape, restricted_total_taxes_data.shape

In [53]:
# fig,ax = plt.subplots(figsize=(25,7))
# sns.boxplot(x = 'trip_pickup_hour',y='tip_amount',data=restricted_tip_amount_data,ax=ax)
# ax.set_title('box plot of tip_amount wrt hour of the day')
# sns.set()
# plt.show()

In [54]:
# fig,ax = plt.subplots(figsize=(25,7))
# sns.boxplot(x = 'trip_dropoff_hour',y='tip_amount',data=restricted_tip_amount_data,ax=ax)
# ax.set_title('box plot of tip_amount wrt hour of the day')
# sns.set()
# plt.show()

Based on tip_amount plot we can see that tip_amount too does not vary much based on hours.

Let's observe total_taxes now

In [55]:
# fig,ax = plt.subplots(figsize=(25,7))
# sns.boxplot(x = 'trip_pickup_hour',y='total_taxes',data=restricted_total_taxes_data,ax=ax)
# ax.set_title('box plot of total_taxes wrt hour of the day')
# sns.set()
# plt.show()

In [56]:
# fig,ax = plt.subplots(figsize=(25,7))
# sns.boxplot(x = 'trip_dropoff_hour',y='total_taxes',data=restricted_total_taxes_data,ax=ax)
# ax.set_title('box plot of total_taxes wrt hour of the day')
# sns.set()
# plt.show()

Now in this plot we can clearly observe that total_taxes change significantly with hour of the day. There are two patterns that we can observe here:<br>
* from the hour 8PM to 5AM the median taxes seem to be a bit higher than other hours, it may be due to some overnight surcharges.
* Evening from 4PM to 7PM have quite variable taxes and is a bit higher than other times, it may be due to higher traffic charges.

Overall the effect of hour of day is most clearly visible on total_taxes. we have two insights about how taxes change with hours
* Overnight charges are applied between 8PM to 5AM
* Evening has higher variability in taxes and the taxes are usually high.

Let's move and explore the distribution of pricing variables with respect to day of week. For this analysis we will be using restricited version of dataset that we built for fare_amount, total_amount, tip_amount and total_taxes.

In [57]:
# # plot of trip_day with fare_amount
# fig,ax = plt.subplots(figsize=(7,7))
# # changes in sns.boxplot x and y
# sns.boxplot(x = 'trip_day',y='fare_amount',data=restricted_fare_amount_data,ax=ax)
# ax.set_title('box plot of fare_amount wrt the day of the week')
# sns.set()
# plt.show()

In [58]:
# fig,ax = plt.subplots(figsize=(7,7))
# sns.boxplot(x = 'trip_day',y='total_amount',data=restricted_total_amount_data,ax=ax)
# ax.set_title('box plot of total_amount wrt the day of the week')
# sns.set()
# plt.show()

In [59]:
# fig,ax = plt.subplots(figsize=(7,7))
# sns.boxplot(x = 'trip_day',y='tip_amount',data=restricted_tip_amount_data,ax=ax)
# ax.set_title('box plot of tip_amount wrt the day of the week')
# sns.set()
# plt.show()

In [60]:
# fig,ax = plt.subplots(figsize=(7,7))
# sns.boxplot(x = 'trip_day',y='total_taxes',data=restricted_total_taxes_data,ax=ax)
# ax.set_title('box plot of total_taxes wrt the day of the week')
# sns.set()
# plt.show()

We can see that pricing overall does not change much with respect to day of week.

*** PRICING VARIABLE EXPLORATION WITH LOCATION OF TRIP ***<br>

Here we will look into the price changes for the most frequent trip pickup locations.

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_33.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 33 ###

# create a new series using value_counts() on 'PULocationID'
pickup_location_value_counts = trip_data['PULocationID'].value_counts()
# show the series
pickup_location_value_counts.head()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_34.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 34 ###

# top 10 frequent pickup locations using .nlargest(10).index
top_10_frequent_pickup_locations = pickup_location_value_counts.nlargest(10).index
top_10_frequent_pickup_locations

In [63]:
# # for loop for plotting box plot of each of the top 10 frequent pickup locations
# for top_pickup_locID in top_10_frequent_pickup_locations:
#     # create the new dataframe for each location using .loc on 'PULocationID' - pickup_locID_dataframe
#     pickup_locID_dataframe = trip_data.loc[trip_data['PULocationID'] == top_pickup_locID]
#     # print the median fare_amount for the top_pickup_locID
#     print('The median fare_amount of trips taken from '+str(top_pickup_locID)+' is '+str(pickup_locID_dataframe['fare_amount'].median()))
#     # fig,ax object
#     fig,ax = plt.subplots(figsize=(6,6))
#     # sns.boxplot of fare_amount from the dataframe pickup_locID_dataframe
#     sns.boxplot(pickup_locID_dataframe['fare_amount'],ax=ax)
#     # set_title
#     ax.set_title('box plot of fare_amount for pickup location '+ str(top_pickup_locID))
#     sns.set()
#     plt.show()

So from above plot we can observe that for one of the most busiest pickup location median fare_amount is quite low in comparison to other busy locations. Though the outliers for pickup location 237 is high.

This could be helpful in adjusting our revenue expectation based on putting our cabs in a given location because just choosing busy pickup locations for higher revenue won't work, we may have to choose locations taking into consideration both busy traffic and higher median fare_amount.

**DURATION EXPLORATION**

Here we will explore the duration of trip exploration with pickup hour of day.

In [64]:
# # plot box plot for duration for different hours of day
# fig,ax = plt.subplots(figsize=(20,7))
# # box plot using sns.boxplot x is 'trip_pickup_hour' and y is 'duration'
# sns.boxplot(x = 'trip_pickup_hour', y='duration',data = trip_data,ax=ax)
# ax.set_title('Box plot of trip_pickup hour with respect to trip duration')
# sns.set()
# plt.show()

Here again due to heavy outliers in duration data we are not able to observe the general graph. we might need to restrict our duration values to within 50min. 

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/new_notebooks/nyc-taxi/small_bench/checkpoints/pre_cell_35.pickle

In [ ]:
%%RecordEvent
%%PandasProfile
### cell 35 ###

# create restricted_duration dataframe with .loc on 'duration' column
restricted_duration= trip_data.loc[trip_data['duration']<50]
restricted_duration.shape

In [66]:
# fig,ax = plt.subplots(figsize=(20,7))
# sns.boxplot(x = 'trip_pickup_hour', y='duration',data = restricted_duration,ax=ax)
# ax.set_title('Box plot of trip_pickup hour with respect to trip duration')
# sns.set()
# plt.show()

Early morning hours of 5AM to 6AM have shorter duration trips

Let's also explore duration with respect to top pickup location.

In [67]:
# # plot box plots of duration for top 10 frequent pickup locations
# for top_pickup_locID in top_10_frequent_pickup_locations:
#     # create the new dataframe for each location using .loc on 'PULocationID' - pickup_locID_dataframe
#     pickup_locID_dataframe = trip_data.loc[trip_data['PULocationID'] == top_pickup_locID]
#     # print the median duration for the top_pickup_locID
#     print('The median trip duration of trips taken from '+str(top_pickup_locID)+' is '+str(pickup_locID_dataframe['duration'].median()))
#     fig,ax = plt.subplots(figsize=(6,6))
#     # sns.boxplot of duration from the dataframe pickup_locID_dataframe
#     sns.boxplot(pickup_locID_dataframe['duration'],ax=ax)
#     # set_title
#     ax.set_title('box plot of duration for pickup location '+ str(top_pickup_locID))
#     sns.set()
#     plt.show()

Here again we can see for the most frequent pickup location 237 the duration value is less in comparison to other pickup locations, though 236 as well has lower duration amount close to 8 min. this might be the reason for less fare_amount as well.

## FINAL RESULTS FROM EDA

Following insights would be useful for our company's product launch in New York
* fare_amount  - most of the fare amount is within 10 dollar value as is shown by the median value. Though there are some significant outliers, the maximum of which is 8000 dollars.


* tip_amount - most of the tip amount is within 1-2 dollar as is shown by the median value. Though again here too we have outliers, the maximum of which is around 450 dollars. 


* duration - most of the values in duration is within 10 minutes range as is shown by the median value. We do have some outliers which are beyond the range of 1000 minutes.


* trip_distance - most of the trip_distance is within 1.55 miles value as is shown by the median. The outlier in this case is quite less.


* Credit card is the most preferred mode of payment followed by cash.


* Peak hour for the pick up and drop off is around evening from 5 to 8. The busiest time is 6PM.


* Weekdays except thursday have heavy taxi uses and among weekends Staturday has taxi uses on the same level as weekdays.


* The busiest location in terms of pickup and dropoff are 161, 236 and 237.


* Four of the busiest routes are - 264-264, 237-236, 236-236, 236-237


* Mostly 1 or 2 passenger avail the cab. Group rides are less common.


* From the hour 8PM to 5AM the median taxes seem to be a bit higher than other hours, it may be due to some overnight surcharges.


* Evening from 4PM to 7PM have quite variable taxes and is a bit higher than other times, it may be due to higher traffic charges.


* We discovered from the dataset that even for the busiest pickup location the median fare_amount is a bit lower than other busier pickup locations. So just choosing busy pickup locations for higher revenue won't work, we may have to choose locations taking into consideration both busy traffic and higher median fare_amount.


* Early morning hours of 5AM to 6AM have shorter duration trips
